In [1]:
# LCEL 문법을 사용하여 chain 을 생성
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import StrOutputParser

# ChatOpenAI 모델을 인스턴스화
model = ChatOpenAI()

# 주어진 토픽에 대한 농담을 요청하는 프롬프트 템플릿을 생성
prompt = PromptTemplate.from_template('{topic}에 대하여 3문장으로 설명해줘.')

# 프롬프트와 모델을 연결하여 대화 체인을 생성
chain = prompt | model | StrOutputParser()

In [2]:
# 주어진 토픽 리스트를 batch 처리하는 함수 호출
chain.batch([{'topic':'ChatGPT'}, {'topic':'Instagram'}])

['ChatGPT는 사용자와의 대화를 통해 자연어 이해 및 생성을 수행하는 인공지능 챗봇이다. 사용자의 질문과 응답을 기반으로 지식을 획득하고 다양한 주제에 대해 대화할 수 있다. ChatGPT는 실시간 대화에서 일상적인 질문부터 복잡한 주제까지 다룰 수 있는 다용도 챗봇이다.',
 'Instagram은 소셜 미디어 플랫폼으로, 사용자들이 사진이나 동영상을 공유하고 다른 사람들과 소통하는 공간이다. 다양한 필터와 편집 기능을 제공하여 사용자들이 자신만의 콘텐츠를 만들 수 있게 도와준다. 또한 해시태그를 통해 관심사나 주제별로 콘텐츠를 검색하고 탐색할 수 있다.']

In [3]:
# 다섯 개의 질문을 처리 -> 최대 3개의 작업을 동시에 처리하도록 설정
chain.batch(
    [
        {'topic':'ChatGPT'},
        {'topic':'Instagram'},
        {'topic':'멀티모달'},
        {'topic':'프로그래밍'},
        {'topic':'머신러닝'}
    ],
    config={'max_concurrency':3},
)

['ChatGPT는 인공 지능 챗봇으로, 자연어 처리 기술을 기반으로 대화 상대와 양방향 대화를 이루는 기능을 제공합니다. 사용자의 입력에 따라 학습하고 성장하여 높은 수준의 상호작용을 제공합니다. 다양한 주제에 대해 대화할 수 있으며, 정보 제공, 질문에 대한 답변, 간단한 연설 생성 등 다양한 기능을 지원합니다.',
 'Instagram은 사진과 동영상을 공유하고 다른 사용자들과 소통할 수 있는 소셜 미디어 플랫폼이다. 사용자들은 팔로우를 통해 친구, 가족, 셀럽 등 다양한 사람들의 활동을 확인하고 자신의 일상을 공유할 수 있다. 인기 있는 해시태그나 필터를 활용하여 보다 멋진 사진을 업로드할 수 있다.',
 '- 멀티모달은 여러 가지 다른 형태의 정보를 조합하여 효과적으로 전달하는 방법입니다.\n- 이는 텍스트, 이미지, 음성, 동영상 등 다양한 매체를 활용하여 사용자에게 다양한 경험을 제공합니다.\n- 멀티모달을 통해 정보의 이해도와 기억력을 높이고, 사용자들에게 더 풍부한 경험을 제공할 수 있습니다.',
 '프로그래밍은 컴퓨터에게 실행할 작업을 지시하는 과정이다. 이를 위해 프로그래밍 언어를 사용하며, 문제 해결 및 알고리즘 작성 능력이 필요하다. 프로그래밍을 통해 다양한 소프트웨어와 애플리케이션을 개발할 수 있으며, 현대 사회의 발전에 큰 영향을 미치고 있다.',
 '머신러닝은 컴퓨터 시스템이 데이터를 분석하고 패턴을 학습하여 예측과 결정을 내릴 수 있는 기술이다. 이를 통해 인간의 개입 없이 시스템이 새로운 데이터에 대한 의사결정을 수행할 수 있다. 주요 알고리즘으로는 지도학습, 비지도학습, 강화 학습이 있다.']

In [4]:
# parallel 병렬성
from langchain_core.runnables import RunnableParallel

# 체인 1 : {country}의 수도를 물어보는 체인
chain1 = (
    PromptTemplate.from_template('{country}의 수도는 어디야?') | model | StrOutputParser()
)

# 체인 2 : {country}의 면적을 물어보는 체인
chain2 = (
    PromptTemplate.from_template('{country}의 면적은 얼마야?') | model | StrOutputParser()
)

# 2개의 체인을 동시에 생성하는 병렬 실행 체인을 생성
combined = RunnableParallel(capital=chain1, area=chain2)

In [5]:
# 체인 1 실행
chain1.invoke({'country':'대한민국'})

'대한민국의 수도는 서울이야.'

In [6]:
# 체인 2 실행
chain2.invoke({'country':'미국'})

'미국의 총 면적은 약 9,826만 제곱킬로미터입니다.'

In [7]:
# 병렬 체인을 실행
combined.invoke({'country':'대한민국'})

{'capital': '대한민국의 수도는 서울이다.', 'area': '대한민국의 면적은 약 100,363㎢ 입니다.'}

In [8]:
chain1.batch([{'country':'대한민국'}, {'country':'미국'}])

['대한민국의 수도는 서울이에요.', '미국의 수도는 워싱턴 D.C. 입니다.']

In [9]:
chain2.batch([{'country':'대한민국'}, {'country':'미국'}])

['대한민국의 총 면적은 약 100,363km² 입니다.', '미국의 총 면적은 9,826,675 km² 입니다.']

In [10]:
# 배치에서의 병렬처리
combined.batch([{'country':'대한민국'}, {'country':'미국'}])

[{'capital': '대한민국의 수도는 서울이다.', 'area': '대한민국의 총 면적은 약 100,363km² 입니다.'},
 {'capital': '미국의 수도는 워싱턴 D.C.입니다.',
  'area': '미국의 총 면적은 대략 9,8백만 제곱 킬로미터 입니다.'}]

In [11]:
from langchain_core.prompts import PromptTemplate
from langchain_openai import ChatOpenAI

prompt = PromptTemplate.from_template('{num}의 10는? ')
llm = ChatOpenAI(temperature=0)

chain = prompt | llm

In [12]:
# chain을 실행
chain.invoke({'num':5})

AIMessage(content='5의 10승은 9765625입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 13, 'prompt_tokens': 14, 'total_tokens': 27, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DwUtnA5jb5wXC4YgBZrtcceQ5ez0e', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--4496989d-e5ee-474c-a7bc-e5d10e21f8ee-0', usage_metadata={'input_tokens': 14, 'output_tokens': 13, 'total_tokens': 27, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [13]:
from langchain_core.runnables import RunnablePassthrough

RunnablePassthrough().invoke({'num':10})

{'num': 10}

In [14]:
runnable_chain = {'num':RunnablePassthrough()} | prompt | ChatOpenAI()

runnable_chain.invoke(10)

AIMessage(content='10의 10제곱은 10,000입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 14, 'prompt_tokens': 14, 'total_tokens': 28, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DwUtpYYzrMxbIUuZZyxUBiWO1O9Nf', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--3769c92f-da51-4c77-b086-c03f7f8d3733-0', usage_metadata={'input_tokens': 14, 'output_tokens': 14, 'total_tokens': 28, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}})

In [15]:
RunnablePassthrough.assign(new_num=lambda x:x['num']*3).invoke({'num':1})
# new_num의 값을 딕셔너리에 추가

{'num': 1, 'new_num': 3}

In [16]:
from langchain_core.runnables import RunnableParallel
runnable = RunnableParallel(
    passed = RunnablePassthrough(), # 입력된 데이터를 그대로 통과시키는 역할
    extra = RunnablePassthrough.assign(mult=lambda x: x['num'] * 3),
    # 입력된 딕셔너리의 'num'키에 해당하는 값에 1을 더한다.
    modified = lambda x: x['num'] + 1,
)

runnable.invoke({'num':1})
# num : 1이 패스돼서 passed, extra, modified를 읽어들임

{'passed': {'num': 1}, 'extra': {'num': 1, 'mult': 3}, 'modified': 2}

In [17]:
chain1 = (
    {'country':RunnablePassthrough()}
    | PromptTemplate.from_template('{country}의 수도는?')
    | ChatOpenAI()
)
chain2 = (
    {'country':RunnablePassthrough()}
    | PromptTemplate.from_template('{country}의 면적은?')
    | ChatOpenAI()
)

In [18]:
combined_chain = RunnableParallel(capital=chain1, area=chain2)
combined_chain.invoke('대한민국')

{'capital': AIMessage(content='서울입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 5, 'prompt_tokens': 18, 'total_tokens': 23, 'completion_tokens_details': {'accepted_prediction_tokens': 0, 'audio_tokens': 0, 'reasoning_tokens': 0, 'rejected_prediction_tokens': 0}, 'prompt_tokens_details': {'audio_tokens': 0, 'cached_tokens': 0}}, 'model_name': 'gpt-3.5-turbo-0125', 'system_fingerprint': None, 'id': 'chatcmpl-DwUtreORtEqnMQORvVa0qFoa8jTxv', 'service_tier': 'default', 'finish_reason': 'stop', 'logprobs': None}, id='run--0818c306-7197-42d7-93b4-f4dd5b12f94f-0', usage_metadata={'input_tokens': 18, 'output_tokens': 5, 'total_tokens': 23, 'input_token_details': {'audio': 0, 'cache_read': 0}, 'output_token_details': {'audio': 0, 'reasoning': 0}}),
 'area': AIMessage(content='대한민국의 총 면적은 약 100,363 km² 입니다.', additional_kwargs={'refusal': None}, response_metadata={'token_usage': {'completion_tokens': 25, 'prompt_tokens': 19, 'total_tokens': 44,